Cross-dataset test: CaptchaSolve30k → SapiMouse (+ Balabit humans)

Does what we found on CaptchaSolve30k hold for **other people, devices and tasks**? Three experiments, with the same features (26 stat features), bot generators (L1–L4 synthetic, L5 replay re-sampled as one device) and metric (recall at FPR ≤ 5 %) as `CaptchaSolve30k_RF_vs_GSFusion_5.ipynb`.

**Data**
- *CaptchaSolve30k* — training source (same humans, bots and splits as v5).
- *SapiMouse* (Antal et al. 2021) — 120 subjects, a 3-min and a 1-min session each, browser-recorded mouse events (timestamp, button, state, x, y). Downloaded from the authors' page.
- *Balabit Mouse Dynamics Challenge* — 10 users, remote-desktop mouse logs; **humans only**, used to measure false positives on yet another population.

**Making them comparable.** External event logs are converted to the CaptchaSolve30k representation: 240 Hz ticks with sample-and-hold (positions are held between input events, as in CaptchaSolve30k), cut into windows of the median CaptchaSolve30k session length, and rescaled from pixels to canvas units with **one factor per dataset chosen label-free** so that the human median speed matches CaptchaSolve30k humans. Bots for SapiMouse are generated with the *same generators* from SapiMouse windows (replay library from a disjoint 20 % of SapiMouse subjects).

**Experiments** (5 seeds, splits always by subject)
- **X1 — transfer without retraining.** Models trained on CaptchaSolve30k (E2: humans + L1–L4). On SapiMouse (a) with the CaptchaSolve30k threshold → does the 5 % FPR hold? (b) with the threshold re-set on half of the SapiMouse *humans* (label-free) → recall per bot level on the other half.
- **X2 — replication within SapiMouse.** E1 (all levels in training) vs E2 (replay unseen). Does the collapse on unseen replay seen on CaptchaSolve30k (RF 0.92 → 0.46) replicate?
- **X3 — Balabit false positives** of the X1 and X2 models at their thresholds.

Session duration and total path length are **excluded** (every external human window has the same length, a bot's does not — see section 9). Models: logistic regression, Random Forest, LightGBM (supervised), Gaussian mixture (human-only). Neural models are left out here.

Licences: SapiMouse and Balabit are public research datasets; cite Antal et al. (2021) and the Balabit Mouse Dynamics Challenge.


In [ ]:
# 1. Setup
%pip install -q datasets lightgbm
import os, math, time, random, warnings, base64, struct, copy
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try: display
except NameError: display = print
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu": print("⚠️  No GPU: Runtime → Change runtime type → T4 GPU, otherwise the neural models take hours.")
from sklearn.mixture import GaussianMixture
import glob, zipfile, urllib.request, io


In [ ]:
# 2. Configuration
N_PER_GAME    = 2000          # human sessions per game (3 games → 6,000); raise for the final run
INCLUDE_TOUCH = False         # touchscreen sessions behave differently; keep them out by default
MIN_TICKS     = 240           # at least 1 s of input
TICK          = 1 / 240       # seconds per physics tick
BOT_SHARE     = 0.25          # bot sessions per level = BOT_SHARE × usable humans (5 levels → 1.25× humans)
FAST          = True          # True: 30 Hz sequences of ≤ 256 steps, fewer epochs (≈ 4× faster); False for the final run
SEQ_STRIDE    = 8 if FAST else 4      # 240 Hz → 30 / 60 Hz for the sequence view
SEQ_MAX       = 256 if FAST else 512  # max sequence steps (≈ 8.5 s)
SEEDS         = [0, 1, 2, 3, 4]
TARGET_FPR    = 0.05
D, EPOCHS, PATIENCE, BATCH, LR = (32, 25, 4, 512, 2e-3) if FAST else (32, 40, 6, 256, 1e-3)
LAMBDA_BAL    = 0.1
REPLAY_RESAMPLE = True        # replay: rebuild the stitched path in continuous time, then re-sample it at the SOURCE human's
                              # update rate and lattice (one device per session, as a real browser replay bot would have)
MATCH_SENSOR  = True          # L1–L4 copy the source human's input update rate and coordinate lattice (replay is left as recorded)
SKIP_SEQ      = None          # None = automatic: skip the LSTM-based models when there is no GPU (they take hours on CPU)
BASE = "/content" if os.path.isdir("/content") else os.path.abspath(".")   # Colab or a local Jupyter
import datetime
RUN_DATE = datetime.date.today().isoformat()   # every result folder, file and zip gets this suffix, e.g. _2026-11-12
def date_files(folder=None):
    """Add _RUN_DATE to every result file in the folder (progress .pkl files keep their name so a run can resume)."""
    folder = folder or OUT
    for p in os.listdir(folder):
        root, ext = os.path.splitext(p)
        if ext == ".pkl" or root.endswith(RUN_DATE): continue
        os.replace(os.path.join(folder, p), os.path.join(folder, f"{root}_{RUN_DATE}{ext}"))
OUT = f"{BASE}/cross_results_{RUN_DATE}"; os.makedirs(OUT, exist_ok=True)
LEVELS = ["linear", "linear_jitter", "bezier_fitts", "bezier_tremor", "human_replay", "human_replay_raw"]
REPLAY = ["human_replay", "human_replay_raw"]          # both are hidden from training in E2
GAMES = ["sheep-herding", "thread-the-needle", "polygon-stacking"]
rng = np.random.default_rng(2026); random.seed(2026)
# cross-dataset
CANVAS     = 200                # bots are clipped to this canvas; set larger for external data later
SAPI_URL   = "http://www.ms.sapientia.ro/~manyi/sapimouse/sapimouse.zip"
BALABIT_URL = "https://github.com/balabit/Mouse-Dynamics-Challenge/archive/refs/heads/master.zip"
MAX_WIN_PER_SESSION = 30        # long sessions (Balabit) would otherwise dominate
MIN_MOVE_SHARE = 0.05           # windows where the pointer moves in < 5 % of ticks are dropped (idle)


In [ ]:
# 3. Load human sessions (streamed — the full file is 5.3 GB)
from datasets import load_dataset
def stream():
    # the repo has held both data.jsonl and train.jsonl; read one file so sessions are not duplicated
    for f in ["train.jsonl", "data.jsonl"]:
        try:
            ds = load_dataset("Capycap-AI/CaptchaSolve30k", data_files=f, split="train", streaming=True)
            next(iter(ds)); print("reading", f); return ds
        except Exception as e:
            print(f"{f}: not usable ({type(e).__name__})")
    return load_dataset("Capycap-AI/CaptchaSolve30k", split="train", streaming=True)

humans, cnt, skipped = [], Counter(), Counter()
t0 = time.time()
for r in stream():
    g = r.get("gameType")
    if g not in GAMES or cnt[g] >= N_PER_GAME:
        if all(cnt[x] >= N_PER_GAME for x in GAMES): break
        continue
    if r.get("touchscreen") and not INCLUDE_TOUCH: skipped["touch"] += 1; continue
    ti = r.get("tickInputs") or []
    if len(ti) < MIN_TICKS: skipped["short"] += 1; continue
    a = np.array([[d["x"], d["y"], float(bool(d["isDown"]))] for d in ti], dtype=np.float32)
    humans.append(dict(game=g, xy=a)); cnt[g] += 1
print(f"{len(humans)} human sessions {dict(cnt)} in {time.time() - t0:.0f}s; skipped {dict(skipped)}")

In [ ]:
# 4. Task skeletons: the waypoints a session must pass through (presses, releases, turns, every 0.4 s)
def skeleton(a, every=96, turn_deg=50, step=12):
    """Indices of waypoints in a tick array a (T×3)."""
    T = len(a); keep = {0, T - 1}
    keep |= set((np.flatnonzero(np.diff(a[:, 2]) != 0) + 1).tolist())           # press / release: first tick of the new state
    keep |= set(range(0, T, every))
    p = a[::step, :2]
    if len(p) > 2:                                                               # sharp turns
        v = np.diff(p, axis=0); ang = np.arctan2(v[:, 1], v[:, 0])
        dang = np.abs((np.diff(ang) + np.pi) % (2 * np.pi) - np.pi)
        moving = (np.linalg.norm(v[:-1], axis=1) > 0.5) & (np.linalg.norm(v[1:], axis=1) > 0.5)
        keep |= set(((np.flatnonzero((dang > np.radians(turn_deg)) & moving) + 1) * step).tolist())
    return np.array(sorted(i for i in keep if i < T))

for h in humans: h["wp"] = skeleton(h["xy"])

In [ ]:
# 5. Splits of the humans: a fixed replay pool, the rest is split per seed later
idx = np.arange(len(humans))
games_arr = np.array([h["game"] for h in humans])
work_idx, replay_idx = train_test_split(idx, test_size=0.20, stratify=games_arr, random_state=7)
print(f"humans for train/val/test: {len(work_idx)} · replay pool (never trained or tested on): {len(replay_idx)}")

# replay library: every waypoint-to-waypoint segment of the pool humans
LIB, LIB_KEY = [], []
for i in replay_idx:
    a, wp = humans[i]["xy"], humans[i]["wp"]
    for s, e in zip(wp[:-1], wp[1:]):
        seg = a[s:e + 1, :2]
        d = float(np.hypot(*(seg[-1] - seg[0])))
        if d < 2 or len(seg) < 4: continue
        th = math.atan2(seg[-1, 1] - seg[0, 1], seg[-1, 0] - seg[0, 0])
        LIB.append(seg); LIB_KEY.append((d, math.cos(th), math.sin(th)))
LIB_KEY = np.array(LIB_KEY); D_MED = np.median(LIB_KEY[:, 0])
KD = cKDTree(np.c_[LIB_KEY[:, 0] / D_MED, LIB_KEY[:, 1], LIB_KEY[:, 2]])
print(f"replay library: {len(LIB):,} human movement segments")

In [ ]:
# 6. Bot generators — same waypoints and button states as the source human, own movement
def ticks_for(dur): return max(2, int(round(dur / TICK)))

def seg_linear(A, B, speed, noise=0.0):
    d = np.hypot(*(B - A)); n = ticks_for(d / speed)
    t = np.linspace(0, 1, n)[:, None]
    p = A + (B - A) * t
    return p + rng.normal(0, noise, p.shape) if noise else p

def seg_bezier(A, B, tremor=False):
    d = np.hypot(*(B - A))
    if d < 0.5: return np.repeat(A[None], 2, 0)
    dur = (0.10 + 0.11 * math.log2(d / 6 + 1)) * math.exp(rng.normal(0, 0.25))
    n = ticks_for(dur); t = np.linspace(0, 1, n)
    if tremor:                                                   # two-phase sub-movement speed profile
        t = np.where(t < 0.7, 0.85 * (t / 0.7) ** 1.2, 0.85 + 0.15 * ((t - 0.7) / 0.3))
    s = t * t * (3 - 2 * t)
    nrm = np.array([-(B - A)[1], (B - A)[0]]) / d
    bend = rng.uniform(-0.25, 0.25) * d
    C1, C2 = A + (B - A) * 0.3 + nrm * bend, A + (B - A) * 0.7 + nrm * bend * 0.5
    u = (1 - s)[:, None]; s2 = s[:, None]
    p = u**3 * A + 3 * u**2 * s2 * C1 + 3 * u * s2**2 * C2 + s2**3 * B
    if tremor:
        f = rng.uniform(8, 12); amp = rng.uniform(0.15, 0.5); ph = rng.uniform(0, 6.3, 2)
        tt = np.arange(n) * TICK
        p = p + amp * np.c_[np.sin(2 * np.pi * f * tt + ph[0]), np.sin(2 * np.pi * f * tt + ph[1])]
    return p

def seg_bezier_tremor(A, B):
    d = np.hypot(*(B - A))
    if d > 10 and rng.random() < 0.2:                            # overshoot past B, then correct back
        O = B + (B - A) / d * rng.uniform(2, 6) + rng.normal(0, 1, 2)
        return np.vstack([seg_bezier(A, O, tremor=True), seg_linear(O, B, rng.uniform(40, 90))[1:]])
    return seg_bezier(A, B, tremor=True)

def seg_replay(A, B):
    d = float(np.hypot(*(B - A)))
    if d < 2: return seg_linear(A, B, 60)
    th = math.atan2(B[1] - A[1], B[0] - A[0])
    _, cand = KD.query([d / D_MED, math.cos(th), math.sin(th)], k=8)
    seg = LIB[int(rng.choice(np.atleast_1d(cand)))]
    v0 = seg[-1] - seg[0]; sc = d / max(np.hypot(*v0), 1e-6)
    rot = th - math.atan2(v0[1], v0[0]); c, s_ = math.cos(rot), math.sin(rot)
    q = (seg - seg[0]) * sc
    return A + np.c_[q[:, 0] * c - q[:, 1] * s_, q[:, 0] * s_ + q[:, 1] * c]

def seg_replay_raw(A, B):
    """Translation only: the nearest human segment by length and direction, NOT rotated or scaled.
    It ends near B rather than on it; the next move starts from wherever it ended."""
    d = float(np.hypot(*(B - A)))
    if d < 2: return seg_linear(A, B, 60)
    th = math.atan2(B[1] - A[1], B[0] - A[0])
    _, cand = KD.query([d / D_MED, math.cos(th), math.sin(th)], k=3)
    seg = LIB[int(rng.choice(np.atleast_1d(cand)))]
    return A + (seg - seg[0])

def sensor_profile(a):
    """(update rate per 240 Hz tick while moving, smallest coordinate step) of a session."""
    p = a[:, :2]; ch = np.any(np.diff(p, axis=0) != 0, axis=1)
    near = np.convolve(ch.astype(float), np.ones(9), "same") > 0
    r = float(ch[near].mean()) if near.any() else 1.0
    d = np.abs(np.diff(p, axis=0)).ravel(); d = d[d > 1e-6]
    return min(max(r, 0.05), 1.0), (float(np.percentile(d, 1)) if len(d) else 0.0)

def apply_sensor(b, r, q):
    """Sample-and-hold at rate r (random phase), then snap to a lattice of step q. The button column is untouched."""
    if r < 0.999:
        k = np.arange(len(b)); ph = rng.random()
        upd = np.floor(k * r + ph) > np.floor((k - 1) * r + ph); upd[0] = True
        b[:, :2] = b[np.maximum.accumulate(np.where(upd, k, 0)), :2]
    if q > 1e-4: b[:, :2] = np.round(b[:, :2] / q) * q
    return b

def resample_replay(b, r, q, max_gap=12):
    """Stitched replay → one device: interpolate between position updates (gaps ≤ max_gap ticks = 50 ms;
    longer gaps are real pauses and stay held), then sample-and-hold at rate r and snap to lattice q."""
    p = b[:, :2].astype(np.float64); t = np.arange(len(b))
    ch = np.r_[True, np.any(np.diff(p, axis=0) != 0, axis=1)]; k = np.flatnonzero(ch)
    if len(k) >= 2:
        prev = k[np.searchsorted(k, t, side="right") - 1]
        nxt = k[np.minimum(np.searchsorted(k, t, side="left"), len(k) - 1)]
        smooth = np.c_[np.interp(t, k, p[k, 0]), np.interp(t, k, p[k, 1])]
        p = np.where(((nxt - prev) <= max_gap)[:, None], smooth, p[prev])
    b = b.copy(); b[:, :2] = p
    r1 = sensor_profile(apply_sensor(b.copy(), r, q))[0]                 # one correction step, as for L1–L4
    return apply_sensor(b, min(1.0, r * r / max(r1, 1e-3)), q)

def make_bot(h, level):
    a, wp = h["xy"], h["wp"]
    speed = rng.uniform(60, 140)
    out = []; cur = a[0, :2].astype(float)
    for s, e in zip(wp[:-1], wp[1:]):
        A, B, down = a[s, :2].astype(float), a[e, :2].astype(float), a[s, 2]
        if level == "human_replay_raw": A = cur
        if level == "linear":          p = seg_linear(A, B, speed)
        elif level == "linear_jitter": p = seg_linear(A, B, speed * math.exp(rng.normal(0, 0.4)), noise=0.3)
        elif level == "bezier_fitts":  p = seg_bezier(A, B)
        elif level == "bezier_tremor": p = seg_bezier_tremor(A, B)
        elif level == "human_replay":  p = seg_replay(A, B)
        else:                          p = seg_replay_raw(A, B)
        cur = p[-1].astype(float)
        out.append(np.c_[p, np.full(len(p), down)])
        if a[e, 2] != down:                                          # press or release at B: dwell, then switch
            pause = {"linear": 0.08, "linear_jitter": rng.uniform(0.02, 0.25)}.get(level, rng.lognormal(math.log(0.08), 0.6))
            n = ticks_for(pause)
            out.append(np.c_[np.repeat(p[-1:], n, 0), np.full(n, a[e, 2])])
    b = np.vstack(out).astype(np.float32)
    if MATCH_SENSOR:
        if "sensor" not in h: h["sensor"] = sensor_profile(a)
        r, q = h["sensor"]
        if level not in REPLAY:                                    # replay keeps the human update timing of its segments
            r1 = sensor_profile(apply_sensor(b.copy(), r, q))[0]   # one correction step: lattice snapping drops updates
            r = min(1.0, r * r / max(r1, 1e-3))
            b = apply_sensor(b, r, q)
        elif REPLAY_RESAMPLE: b = resample_replay(b, r, q)       # replay: one device rate + lattice (see 9c/9d)
    b[:, :2] = np.clip(b[:, :2], 0, CANVAS)
    return b

bots = []
for lv in LEVELS:
    src = rng.choice(work_idx, size=int(BOT_SHARE * len(work_idx)), replace=False)
    for i in src:
        bots.append(dict(game=humans[i]["game"], xy=make_bot(humans[i], lv), level=lv, src=int(i)))
print(f"{len(bots)} bot sessions:", dict(Counter(b["level"] for b in bots)))

In [ ]:
# 7. Features for the three views
def dyn(a):
    p, dn = a[:, :2].astype(np.float64), a[:, 2]
    v = np.diff(p, axis=0) / TICK; sp = np.hypot(v[:, 0], v[:, 1])
    acc = np.diff(sp) / TICK if len(sp) > 1 else np.zeros(1)
    jerk = np.diff(acc) / TICK if len(acc) > 1 else np.zeros(1)
    mv = sp > 1.0
    ang = np.arctan2(v[:, 1], v[:, 0]); dang = np.abs((np.diff(ang) + np.pi) % (2 * np.pi) - np.pi)
    dang = dang[mv[1:] & mv[:-1]] if len(dang) else np.zeros(1)
    plen = sp.sum() * TICK; disp = np.hypot(*(p[-1] - p[0]))
    idle = ~mv; runs = np.diff(np.r_[0, idle.astype(int), 0]); n_pause = int((runs == 1).sum())
    presses = int((np.diff(dn) > 0).sum()); dur = len(a) * TICK
    hist = np.histogram(ang[mv], bins=8, range=(-np.pi, np.pi))[0] + 1e-9; hist = hist / hist.sum()
    # local speed minima per second = sub-movements
    s_sm = np.convolve(sp, np.ones(9) / 9, "same")
    minima = int(((s_sm[1:-1] < s_sm[:-2]) & (s_sm[1:-1] < s_sm[2:]) & (s_sm[1:-1] < np.percentile(s_sm, 60))).sum()) if len(s_sm) > 2 else 0
    # tremor band energy share (8–12 Hz) of the speed signal
    if len(sp) > 64:
        F = np.abs(np.fft.rfft(sp - sp.mean())) ** 2; fr = np.fft.rfftfreq(len(sp), TICK)
        trem = F[(fr >= 8) & (fr <= 12)].sum() / (F[fr > 0.5].sum() + 1e-9)
    else: trem = 0.0
    q = lambda x, k: float(np.percentile(x, k)) if len(x) else 0.0
    return [math.log1p(dur), math.log1p(plen), disp / (plen + 1e-9),
            sp.mean(), sp.std(), q(sp, 10), q(sp, 50), q(sp, 90), sp.max(),
            np.abs(acc).mean(), acc.std(), np.abs(jerk).mean(), jerk.std(),
            dang.mean() if len(dang) else 0, dang.std() if len(dang) else 0,
            idle.mean(), n_pause / dur, dn.mean(), presses / dur,
            float(-(hist * np.log(hist)).sum()), minima / dur, trem,
            float((np.abs(np.diff(p, axis=0)).sum(1) == 0).mean()),       # exactly repeated positions
            np.ptp(p[:, 0]), np.ptp(p[:, 1]),
            float(np.corrcoef(sp[:-1], sp[1:])[0, 1]) if len(sp) > 3 and sp.std() > 0 else 0.0]

STAT_NAMES = ["log_dur", "log_path", "straightness", "speed_mean", "speed_std", "speed_p10", "speed_p50",
              "speed_p90", "speed_max", "acc_mean", "acc_std", "jerk_mean", "jerk_std", "turn_mean", "turn_std",
              "idle_share", "pauses_per_s", "down_share", "presses_per_s", "direction_entropy",
              "submovements_per_s", "tremor_share", "repeat_share", "x_range", "y_range", "speed_autocorr"]

def seq_view(a):
    s = a[::SEQ_STRIDE]
    d = np.diff(s[:, :2], axis=0)
    x = np.c_[d / 5.0, s[1:, 2]].astype(np.float32)[:SEQ_MAX]
    return x if len(x) else np.zeros((1, 3), np.float32)

def graph_view(a):
    s = a[::SEQ_STRIDE, :2]; d = np.diff(s, axis=0); mv = np.hypot(d[:, 0], d[:, 1]) > 0.3
    dirs = ((np.arctan2(d[:, 1], d[:, 0]) + np.pi) / (2 * np.pi) * 8).astype(int) % 8
    dm = dirs[mv]; M = np.ones((8, 8))
    for x, y in zip(dm[:-1], dm[1:]): M[x, y] += 1
    M = M / M.sum(1, keepdims=True)
    cells = (np.clip(s[:, 0], 0, 199.9) // 20 * 10 + np.clip(s[:, 1], 0, 199.9) // 20).astype(int)
    ch = cells[np.r_[True, cells[1:] != cells[:-1]]]
    tr = Counter(zip(ch[:-1], ch[1:])); tot = max(sum(tr.values()), 1)
    pt = np.array(list(tr.values())) / tot if tr else np.ones(1)
    seen, rev = set(), 0
    for c in ch: rev += c in seen; seen.add(c)
    return np.r_[M.ravel(), float(-(pt * np.log(pt)).sum()), rev / max(len(ch), 1), len(seen) / 100.0].astype(np.float32)

GRAPH_NAMES = [f"dir{i}->{j}" for i in range(8) for j in range(8)] + ["cell_transition_entropy", "cell_revisit_ratio", "unique_cells"]
ALL = humans + bots
Y = np.array([0] * len(humans) + [1] * len(bots))
LEVEL = np.array(["human"] * len(humans) + [b["level"] for b in bots])
ORIGIN = np.array(list(range(len(humans))) + [b["src"] for b in bots])     # source human of each bot
t0 = time.time()
X_STAT = np.nan_to_num(np.array([dyn(s["xy"]) for s in ALL], dtype=np.float32))
SEQS = [seq_view(s["xy"]) for s in ALL]
X_GRAPH = np.array([graph_view(s["xy"]) for s in ALL], dtype=np.float32)
print(f"features for {len(ALL)} sessions in {time.time() - t0:.0f}s · stat {X_STAT.shape[1]} · graph {X_GRAPH.shape[1]}")

In [ ]:
# 8. External data: download, parse event logs, convert to 240 Hz ticks, cut into windows
def fetch(url, dest):
    if os.path.exists(dest): return dest
    print("downloading", url)
    try:
        urllib.request.urlretrieve(url, dest); return dest
    except Exception as e:
        print(f"⚠️  download failed ({e}). Download it manually to {dest} and re-run this cell."); raise

def unzip(zpath, dest):
    if not os.path.isdir(dest):
        with zipfile.ZipFile(zpath) as z: z.extractall(dest)
    return dest

def read_log(path):
    """Event log → (t seconds, x, y, down). Handles SapiMouse and Balabit layouts, with or without a header."""
    first = open(path, errors="ignore").readline()
    if any(c.isalpha() for c in first.split(",")[0]):
        df = pd.read_csv(path); df.columns = [c.strip().lower() for c in df.columns]
        ts = "client timestamp" if "client timestamp" in df.columns else [c for c in df.columns if "timestamp" in c][0]
        df = df.rename(columns={ts: "t"})
    else:
        df = pd.read_csv(path, header=None)
        df.columns = ["t", "button", "state", "x", "y"][:df.shape[1]] if df.shape[1] == 5 else \
                     ["rt", "t", "button", "state", "x", "y"][:df.shape[1]]
    df = df[["t", "button", "state", "x", "y"]].copy()
    for c in ["t", "x", "y"]: df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["t", "x", "y"]); df = df[(df.x.abs() < 20000) & (df.y.abs() < 20000)]     # Balabit has 65535 artefacts
    df = df.sort_values("t", kind="stable")
    t = df.t.to_numpy(float); dt = np.diff(t); dt = dt[dt > 0]
    if len(dt) and np.median(dt) > 0.5: t = t / 1000.0                                        # milliseconds → seconds
    down, cur = np.zeros(len(df)), 0.0
    for i, (b, s) in enumerate(zip(df.button.astype(str).str.lower(), df.state.astype(str).str.lower())):
        if s == "pressed" and b in ("left", "right"): cur = 1.0
        elif s == "released": cur = 0.0
        elif s == "drag": cur = 1.0
        down[i] = cur
    return t, df.x.to_numpy(float), df.y.to_numpy(float), down

def to_ticks(t, x, y, d):
    """Sample-and-hold at 240 Hz: each tick takes the last event at or before it."""
    if len(t) < 2 or t[-1] - t[0] < 1: return None
    tk = t[0] + np.arange(int((t[-1] - t[0]) / TICK) + 1) * TICK
    i = np.searchsorted(t, tk, side="right") - 1
    return np.c_[x[i], y[i], d[i]].astype(np.float32)

def windows(a, win):
    out = []
    for s in range(0, len(a) - win + 1, win):
        w = a[s:s + win].copy()
        if np.any(np.diff(w[:, :2], axis=0) != 0, axis=1).mean() < MIN_MOVE_SHARE: continue
        w[:, :2] -= w[:, :2].min(0) - 10                          # translate near the origin (scale is set later)
        out.append(w)
        if len(out) >= MAX_WIN_PER_SESSION: break
    return out

WIN = int(np.median([len(h["xy"]) for h in humans]))                  # median CaptchaSolve30k session length
print(f"window length: {WIN} ticks = {WIN * TICK:.1f} s")

def load_external(files, subject_of):
    ws = []
    for f in files:
        try: a = to_ticks(*read_log(f))
        except Exception as e: print("skip", f, type(e).__name__); continue
        if a is None: continue
        ws += [dict(xy=w, subj=subject_of(f), file=os.path.basename(f)) for w in windows(a, WIN)]
    return ws

sapi_dir = unzip(fetch(SAPI_URL, f"{BASE}/sapimouse.zip"), f"{BASE}/sapimouse")
sapi_files = [f for f in glob.glob(f"{sapi_dir}/**/*.csv", recursive=True) if "session" in os.path.basename(f)]
SAPI = load_external(sapi_files, lambda f: os.path.basename(os.path.dirname(f)))
bal_dir = unzip(fetch(BALABIT_URL, f"{BASE}/balabit.zip"), f"{BASE}/balabit")
bal_files = [f for f in glob.glob(f"{bal_dir}/**/training_files/*/*", recursive=True) if os.path.isfile(f)]
BAL = load_external(bal_files, lambda f: os.path.basename(os.path.dirname(f)))
print(f"SapiMouse: {len(sapi_files)} files → {len(SAPI)} windows from {len(set(w['subj'] for w in SAPI))} subjects")
print(f"Balabit:   {len(bal_files)} files → {len(BAL)} windows from {len(set(w['subj'] for w in BAL))} users")


In [ ]:
# 9. Rescale (label-free), build SapiMouse bots with the same generators, compute features
def move_speed(a, k=8):
    """Median speed over ~33 ms steps while moving — robust to how often the device reports positions."""
    d = np.hypot(*(a[k:, :2] - a[:-k, :2]).T) / (k * TICK); d = d[d > 0]
    return float(np.median(d)) if len(d) else np.nan
med_cs = np.nanmedian([move_speed(h["xy"]) for h in humans])
def rescale(ws, name):
    raw = np.array([move_speed(w["xy"]) for w in ws])
    s = float(np.nanmedian(raw) / med_cs)
    for w in ws: w["xy"][:, :2] /= s
    print(f"{name}: {s:.2f} px per canvas unit (human median speed matched to CaptchaSolve30k)")
    return s
SCALE = {"SapiMouse": rescale(SAPI, "SapiMouse"), "Balabit": rescale(BAL, "Balabit")}

ext_h = [dict(game="sapimouse", xy=w["xy"], subj=w["subj"]) for w in SAPI]
for h in ext_h: h["wp"] = skeleton(h["xy"])
subjects = np.array(sorted(set(h["subj"] for h in ext_h)))
pool_subj = set(np.random.default_rng(7).choice(subjects, size=max(1, len(subjects) // 5), replace=False))
work_subj = np.array([s for s in subjects if s not in pool_subj])
SUBJ = np.array([h["subj"] for h in ext_h])
pool_i, work_i = np.flatnonzero(np.isin(SUBJ, list(pool_subj))), np.flatnonzero(~np.isin(SUBJ, list(pool_subj)))

# swap the replay library to SapiMouse pool subjects (the CaptchaSolve30k bots are already generated)
LIB, LIB_KEY = [], []
for i in pool_i:
    a, wp = ext_h[i]["xy"], ext_h[i]["wp"]
    for s, e in zip(wp[:-1], wp[1:]):
        seg = a[s:e + 1, :2]; d = float(np.hypot(*(seg[-1] - seg[0])))
        if d < 2 or len(seg) < 4: continue
        th = math.atan2(seg[-1, 1] - seg[0, 1], seg[-1, 0] - seg[0, 0])
        LIB.append(seg); LIB_KEY.append((d, math.cos(th), math.sin(th)))
LIB_KEY = np.array(LIB_KEY); D_MED = np.median(LIB_KEY[:, 0])
KD = cKDTree(np.c_[LIB_KEY[:, 0] / D_MED, LIB_KEY[:, 1], LIB_KEY[:, 2]])
CANVAS = 1e9                                                   # external windows are not limited to a 200-unit canvas
ext_b = []
for lv in LEVELS:
    for i in rng.choice(work_i, size=int(BOT_SHARE * len(work_i)), replace=False):
        ext_b.append(dict(xy=make_bot(ext_h[i], lv), level=lv, src=int(i)))
print(f"SapiMouse: {len(work_i)} human windows ({len(work_subj)} subjects) · {len(pool_i)} pool windows · {len(ext_b)} bots",
      dict(Counter(b["level"] for b in ext_b)))

S_ALL = [ext_h[i] for i in work_i] + ext_b
S_Y = np.array([0] * len(work_i) + [1] * len(ext_b))
S_LEVEL = np.array(["human"] * len(work_i) + [b["level"] for b in ext_b])
S_SUBJ = np.array([SUBJ[i] for i in work_i] + [SUBJ[b["src"]] for b in ext_b])
t0 = time.time()
S_X = np.nan_to_num(np.array([dyn(s["xy"]) for s in S_ALL], dtype=np.float32))
B_X = np.nan_to_num(np.array([dyn(w["xy"]) for w in BAL], dtype=np.float32))
print(f"features: SapiMouse {S_X.shape} · Balabit {B_X.shape} in {time.time() - t0:.0f}s")

# Every human window is exactly WIN ticks long, while a bot's length depends on its own movement timing, so
# session duration (and total path length, which grows with it) would identify bots trivially. Drop both.
DROP = ["log_dur", "log_path"]
FEATS = [j for j, n in enumerate(STAT_NAMES) if n not in DROP]; F_NAMES = [STAT_NAMES[j] for j in FEATS]
X_STAT_F, S_XF, B_XF = X_STAT[:, FEATS], S_X[:, FEATS], B_X[:, FEATS]
print("duration (s) by level — why duration is dropped:")
display(pd.Series(np.expm1(S_X[:, 0])).groupby(S_LEVEL).describe()[["min", "50%", "max"]].round(2))
# artefact check: best single remaining feature, humans vs each SapiMouse bot level (direction-free AUC)
hum = S_Y == 0; art = {}
for lv in LEVELS:
    b = S_LEVEL == lv; y_ = np.r_[np.zeros(hum.sum()), np.ones(b.sum())]
    au = {n: roc_auc_score(y_, np.r_[S_XF[hum, j], S_XF[b, j]]) for j, n in enumerate(F_NAMES)}
    au = {n: max(v, 1 - v) for n, v in au.items()}; k = max(au, key=au.get); art[lv] = (k, round(au[k], 3))
print("best single feature per level (AUC ≥ 0.95 = suspect generator artefact):", art)

# domain check: how far are the external humans from CaptchaSolve30k humans, feature by feature?
cs_h = X_STAT[LEVEL == "human"]
shift = pd.DataFrame({"CaptchaSolve30k": np.median(cs_h, 0), "SapiMouse": np.median(S_X[S_Y == 0], 0),
                      "Balabit": np.median(B_X, 0)}, index=STAT_NAMES).round(3)
display(shift)


In [ ]:
# 10. Models and thresholds
def thr_from_humans(s_h):
    h = np.sort(s_h); k = int(np.floor(TARGET_FPR * len(h)))
    return float(np.nextafter(h[len(h) - k - 1], np.inf))

def fit_models(X, y, seed):
    """Supervised models on (X, y) and a human-only GMM on the humans in X. Returns name → score function."""
    sc = StandardScaler().fit(X); Xh = X[y == 0]; sh = StandardScaler().fit(Xh)
    lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(sc.transform(X), y)
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=seed).fit(X, y)
    lg = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, class_weight="balanced",
                            random_state=seed, verbose=-1).fit(X, y)
    gm = GaussianMixture(n_components=8, covariance_type="full", reg_covar=1e-3, random_state=seed).fit(sh.transform(Xh))
    return {"Logistic regression": lambda Z: lr.predict_proba(sc.transform(Z))[:, 1],
            "Random Forest":       lambda Z: rf.predict_proba(Z)[:, 1],
            "LightGBM":            lambda Z: lg.predict_proba(Z)[:, 1],
            "GMM (human-only)":    lambda Z: -gm.score_samples(sh.transform(Z))}

rows = []
def record(exp, seed, model, thr_src, dataset, score, thr, y, lv):
    pred = score >= thr
    row = dict(exp=exp, seed=seed, model=model, threshold=thr_src, dataset=dataset,
               ROC_AUC=roc_auc_score(y, score) if len(np.unique(y)) == 2 else np.nan,
               FPR=float(pred[y == 0].mean()), recall_all=float(pred[y == 1].mean()) if (y == 1).any() else np.nan)
    for L in LEVELS: row[L] = float(pred[lv == L].mean()) if (lv == L).any() else np.nan
    rows.append(row)


In [ ]:
# 11. X1 — transfer: trained on CaptchaSolve30k (E2), applied to SapiMouse and Balabit without retraining
t0 = time.time()
for seed in SEEDS:
    rest, te_h = train_test_split(work_idx, test_size=0.2, stratify=games_arr[work_idx], random_state=seed)
    tr_h, va_h = train_test_split(rest, test_size=0.2, stratify=games_arr[rest], random_state=seed)
    tr_i = np.flatnonzero(np.isin(ORIGIN, tr_h) & ~np.isin(LEVEL, REPLAY))
    va_hum = np.flatnonzero(np.isin(ORIGIN, va_h) & (LEVEL == "human"))
    M = fit_models(X_STAT_F[tr_i], Y[tr_i], seed)
    calib_s = set(np.random.default_rng(seed).choice(work_subj, size=len(work_subj) // 2, replace=False))
    cal = np.isin(S_SUBJ, list(calib_s)); ev = ~cal
    for name, f in M.items():
        thr_cs = thr_from_humans(f(X_STAT_F[va_hum]))
        s_S, s_B = f(S_XF), f(B_XF)
        record("X1", seed, name, "CaptchaSolve30k", "SapiMouse", s_S[ev], thr_cs, S_Y[ev], S_LEVEL[ev])
        record("X1", seed, name, "SapiMouse humans (re-set)", "SapiMouse", s_S[ev], thr_from_humans(s_S[cal & (S_Y == 0)]),
               S_Y[ev], S_LEVEL[ev])
        record("X1", seed, name, "CaptchaSolve30k", "Balabit", s_B, thr_cs, np.zeros(len(s_B), int), np.array(["human"] * len(s_B)))
    print(f"X1 seed {seed} done ({time.time() - t0:.0f}s)")


In [ ]:
# 12. X2 — replication within SapiMouse: E1 (all levels seen) vs E2 (replay unseen); X3 — Balabit FPR of these models
t0 = time.time()
for seed in SEEDS:
    rest, te_s = train_test_split(work_subj, test_size=0.2, random_state=seed)
    tr_s, va_s = train_test_split(rest, test_size=0.25, random_state=seed)
    side = np.select([np.isin(S_SUBJ, tr_s), np.isin(S_SUBJ, va_s), np.isin(S_SUBJ, te_s)], [0, 1, 2], -1)
    for exp in ["X2-E1", "X2-E2"]:
        tr = side == 0; va = (side == 1) & (S_Y == 0); te = side == 2
        if exp == "X2-E2": tr &= ~np.isin(S_LEVEL, REPLAY)
        M = fit_models(S_XF[tr], S_Y[tr], seed)
        for name, f in M.items():
            thr = thr_from_humans(f(S_XF[va]))
            record(exp, seed, name, "SapiMouse", "SapiMouse", f(S_XF[te]), thr, S_Y[te], S_LEVEL[te])
            if exp == "X2-E2":
                s_B = f(B_XF)
                record("X3", seed, name, "SapiMouse", "Balabit", s_B, thr, np.zeros(len(s_B), int), np.array(["human"] * len(s_B)))
    print(f"X2/X3 seed {seed} done ({time.time() - t0:.0f}s)")


In [ ]:
# 13. Results
res = pd.DataFrame(rows); res["replay"] = res[REPLAY].mean(1); res["L1_L4_min"] = res[LEVELS[:4]].min(1)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
key = ["exp", "threshold", "dataset", "model"]
summary = res.groupby(key, sort=False)[["ROC_AUC", "FPR"] + LEVELS + ["replay", "L1_L4_min"]].mean().round(3)
display(summary)
sd = res.groupby(key, sort=False)[["FPR", "replay"]].std().round(3).add_suffix("_sd"); display(sd)

print("\nX2 — does the collapse on unseen replay replicate? (replay recall, E1 → E2)")
x2 = res[res.exp.str.startswith("X2")].groupby(["model", "exp"])["replay"].mean().unstack().round(3)
x2["drop"] = (x2["X2-E1"] - x2["X2-E2"]).round(3); display(x2)
print("False positives on other humans (target 0.05):")
display(res[res.dataset == "Balabit"].groupby(["exp", "model"])["FPR"].agg(["mean", "std"]).round(3))

res.to_csv(f"{OUT}/cross_raw.csv", index=False); summary.to_csv(f"{OUT}/cross_table.csv"); shift.to_csv(f"{OUT}/domain_shift.csv")
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
x2.drop(columns="drop").plot(kind="bar", ax=ax[0]); ax[0].set_title("SapiMouse: replay recall, seen (E1) vs unseen (E2)")
ax[0].set_ylim(0, 1.02); ax[0].tick_params(axis="x", rotation=15); ax[0].set_ylabel(f"recall at FPR ≤ {TARGET_FPR:.0%}")
fp = res[res.threshold != "SapiMouse humans (re-set)"].groupby(["dataset", "exp", "model"])["FPR"].mean().unstack(0)
fp.plot(kind="bar", ax=ax[1]); ax[1].axhline(TARGET_FPR, ls="--", c="k"); ax[1].set_title("Human false-positive rate at the source threshold")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_cross.png", dpi=200); plt.show()
date_files()
with zipfile.ZipFile(f"{BASE}/cross_results_{RUN_DATE}.zip", "w") as z:
    for p in os.listdir(OUT): z.write(f"{OUT}/{p}", p)
print(f"saved {BASE}/cross_results_{RUN_DATE}.zip")
